In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("retail_sales_dataset.csv")

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nDataset Preview:")
display(df.head())

Dataset Shape: (1000, 9)

Column Names:
['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']

Missing Values:
Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

Duplicate Rows: 0

Dataset Preview:


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [3]:
completeness = (df.notnull().sum().sum() / df.size) * 100

accuracy_errors = (
    (df["Age"] < 18) |
    (df["Age"] > 100) |
    (df["Quantity"] <= 0) |
    (df["Price per Unit"] < 0) |
    (df["Total Amount"] < 0)
).sum()

consistency_errors = (
    df["Total Amount"] != df["Quantity"] * df["Price per Unit"]
).sum()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
timeliness_errors = df["Date"].isna().sum()

duplicate_ids = df["Transaction ID"].duplicated().sum()

valid_categories = ["Beauty", "Clothing", "Electronics"]
invalid_categories = (~df["Product Category"].isin(valid_categories)).sum()


print("DAMA DATA QUALITY ASSESSMENT")
print("--------------------------------")
print("Completeness:", round(completeness, 2), "%")
print("Accuracy errors:", accuracy_errors)
print("Consistency errors:", consistency_errors)
print("Timeliness errors:", timeliness_errors)
print("Duplicate Transaction IDs:", duplicate_ids)
print("Invalid Product Categories:", invalid_categories)

DAMA DATA QUALITY ASSESSMENT
--------------------------------
Completeness: 100.0 %
Accuracy errors: 0
Consistency errors: 0
Timeliness errors: 0
Duplicate Transaction IDs: 0
Invalid Product Categories: 0


In [4]:
import great_expectations as gx

context = gx.get_context()

data_source = context.data_sources.add_pandas(
    name="retail_sales_source"
)
asset = data_source.add_dataframe_asset(
    name="retail_sales"
)

batch_definition = asset.add_batch_definition_whole_dataframe(
    "retail_sales_batch"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

print("Great Expectations connected successfully!")

Great Expectations connected successfully!


In [5]:
suite = gx.ExpectationSuite(name="retail_sales_suite")

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="Transaction ID")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="Customer ID")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column="Transaction ID")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Age", min_value=18, max_value=100
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Quantity", min_value=1
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Price per Unit", min_value=0
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Total Amount", min_value=0
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="Product Category",
        value_set=["Beauty", "Clothing", "Electronics"]
    )
)

context.suites.add(suite)

print("Created 8 data quality expectations successfully!")

Created 8 data quality expectations successfully!


In [6]:
validation_definition = gx.ValidationDefinition(
    data=batch_definition,
    suite=suite,
    name="retail_sales_validation"
)

context.validation_definitions.add(validation_definition)

results = validation_definition.run(
    batch_parameters={"dataframe": df}
)

print("Validation Success:", results.success)
print("Total Expectations:", len(results.results))
print("Successful Expectations:", sum(r.success for r in results.results))
print("Failed Expectations:", sum(not r.success for r in results.results))

Calculating Metrics:   0%|          | 0/53 [00:00<?, ?it/s]

Validation Success: True
Total Expectations: 8
Successful Expectations: 8
Failed Expectations: 0


In [7]:
df_bad = df.copy()

df_bad.loc[0, "Customer ID"] = None

df_bad.loc[1, "Age"] = 150

df_bad.loc[2, "Product Category"] = "Unknown"

print("Three intentional errors added!")

display(df_bad.head(3))

Three intentional errors added!


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,NaN,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,150,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Unknown,1,30,30


In [8]:
bad_results = validation_definition.run(
    batch_parameters={"dataframe": df_bad}
)

print("Validation Success:", bad_results.success)
print("Total Expectations:", len(bad_results.results))
print("Successful Expectations:", sum(r.success for r in bad_results.results))
print("Failed Expectations:", sum(not r.success for r in bad_results.results))

for result in bad_results.results:
    if not result.success:
        print("\nFailed Check:", result.expectation_config.type)

Calculating Metrics:   0%|          | 0/53 [00:00<?, ?it/s]

Validation Success: False
Total Expectations: 8
Successful Expectations: 5
Failed Expectations: 3

Failed Check: expect_column_values_to_not_be_null

Failed Check: expect_column_values_to_be_between

Failed Check: expect_column_values_to_be_in_set


In [9]:
import json
from pathlib import Path
from IPython.display import display, HTML

report_data = bad_results.describe()

Path("validation_results.json").write_text(
    json.dumps(report_data, indent=4, default=str)
)

html = f"""
<html>
<head>
    <title>Data Quality Validation Report</title>
</head>
<body>
    <h1>Retail Sales - Data Quality Report</h1>
    <h2>Validation Summary</h2>
    <p>Validation Success: {bad_results.success}</p>
    <p>Total Expectations: {len(bad_results.results)}</p>
    <p>Passed: 5</p>
    <p>Failed: 3</p>
    <h2>Failed Checks</h2>
    <ul>
"""

for result in bad_results.results:
    if not result.success:
        html += f"<li>{result.expectation_config.type}</li>"

html += """
    </ul>
</body>
</html>
"""

Path("validation_report.html").write_text(html, encoding="utf-8")

display(HTML(html))

print("HTML validation report created successfully!")

HTML validation report created successfully!


In [10]:
pii_classification = pd.DataFrame({
    "Column": [
        "Transaction ID",
        "Customer ID",
        "Gender",
        "Age",
        "Date"
    ],
    "Classification": [
        "Internal Identifier",
        "Pseudonymous Personal Data",
        "Personal Data",
        "Personal Data",
        "Potentially Personal Data"
    ],
    "Protection Method": [
        "Restrict access",
        "Mask or tokenize",
        "Limit access",
        "Convert into age groups",
        "Use aggregated dates where possible"
    ]
})

display(pii_classification)

,Column,Classification,Protection Method
0,Transaction ID,Internal Identifier,Restrict access
1,Customer ID,Pseudonymous Personal Data,Mask or tokenize
2,Gender,Personal Data,Limit access
3,Age,Personal Data,Convert into age groups
4,Date,Potentially Personal Data,Use aggregated dates where possible


In [11]:
scorecard = pd.DataFrame({
    "DAMA Dimension": [
        "Completeness",
        "Accuracy",
        "Consistency",
        "Timeliness",
        "Uniqueness",
        "Validity"
    ],
    "Score": [
        "100%",
        "100%",
        "100%",
        "100%",
        "100%",
        "100%"
    ],
    "Status": ["Pass"] * 6,
    "Recommendation": [
        "Continue checking missing values",
        "Verify data against source records",
        "Maintain consistent data formats",
        "Monitor data freshness regularly",
        "Prevent duplicate transaction IDs",
        "Validate allowed values and ranges"
    ]
})

display(scorecard)

,DAMA Dimension,Score,Status,Recommendation
0,Completeness,100%,Pass,Continue checking missing values
1,Accuracy,100%,Pass,Verify data against source records
2,Consistency,100%,Pass,Maintain consistent data formats
3,Timeliness,100%,Pass,Monitor data freshness regularly
4,Uniqueness,100%,Pass,Prevent duplicate transaction IDs
5,Validity,100%,Pass,Validate allowed values and ranges
